In [ ]:
# imports

import pandas as pd
import plotly.express as px

In [ ]:
from pathlib import Path

# Folder holding one exported CSV per listener
DATA_DIR = Path("data")

# Encodings to try in order; utf-8-sig also strips the BOM some exports have
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]


def read_csv_any_encoding(path):
    """Read a CSV, trying each encoding until one works."""
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            # Wrong encoding for this file, try the next one
            continue
    raise ValueError(f"Could not read {path} with any of {ENCODINGS}")


# Load every CSV in the data folder and tag its rows with the listener
frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = read_csv_any_encoding(path)
    # Listener name is the first word of the file name,
    # e.g. "Logan Exported Data.csv" -> "Logan"
    df["Listener"] = path.stem.split()[0]
    frames.append(df)

# Stack all listeners into one DataFrame
data = pd.concat(frames, ignore_index=True)

# Quick sanity check: total size and rows per listener
print(data.shape)
print(data["Listener"].value_counts())
data.head()

### Data Cleaning

In [ ]:
# identify missing values in each column
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# handle the 1 missing values row first

# display the row with missing values - NOT THE GENRES MISSING VALUES
data[data.isnull().any(axis=1) & data['Mode'].isnull()]

In [ ]:
# drop track uri spotify:episode:3bX59hUSv3Lh3vBxmmmcNP due to bad data in the row
data = data[data['Track URI'] != 'spotify:episode:3bX59hUSv3Lh3vBxmmmcNP']

In [ ]:
# rerun missing values check

missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# drop added by column as this is an unneeded column for our analysis
data = data.drop(columns=['Added By'])

In [ ]:
# rerun missing values check
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# inspect missing values in the genres column
data[data['Genres'].isnull()]

# add a flag for missing genres
data['Missing Genres'] = data['Genres'].isnull()


def to_list(value):
    """Split a comma-separated cell into a clean list; missing becomes []."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",") if part.strip()]


# Artist Name(s) and Genres each hold several values per track, so store them
# as lists, and keep a text version for hover labels and display
for column in ["Artist Name(s)", "Genres"]:
    data[column] = data[column].apply(to_list)
    data[column + " Text"] = data[column].apply(", ".join)

data.head()

In [ ]:
data.groupby("Listener").size()

In [ ]:
data.columns.tolist()

In [ ]:
data.shape

In [ ]:
data.groupby("Listener")["Missing Genres"].sum()

In [ ]:
(
    data.explode("Genres")
    .groupby("Listener")["Genres"]
    .value_counts(dropna=False)
    .groupby(level=0)
    .head(20)
)

In [ ]:
features = [
    "Valence",
    "Energy",
    "Danceability",
    "Acousticness",
    "Tempo",
    "Loudness",
    "Popularity"
]

data.groupby("Listener")[features].describe()

In [ ]:
data.head()

In [ ]:
fig = px.scatter(
    data,
    x="Valence",
    y="Energy",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Valence vs Energy of Songs"
)

fig.show()

The graph shows that songs with higher valence often also have higher energy, but there is still a lot of variation. This suggests that valence and energy are somewhat related, but they are measuring different characteristics of a song.

In [ ]:
data.groupby("Listener")[features].corr()

The results show that energy and loudness have the strongest relationship, so songs with more energy also tend to be louder. Energy also decreases as acousticness increases. Valence and energy have a positive but moderate relationship, which supports what we saw in the scatterplot. Popularity does not seem to have a strong relationship with the other audio features.

In [ ]:
fig = px.scatter(
    data,
    x="Energy",
    y="Loudness",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s) Text", "Genres Text"],
    title="Energy vs Loudness of Songs"
)

fig.show()

I made this graph because the correlation results showed that energy and loudness had the strongest relationship in our dataset. The graph makes that relationship easier to see: as songs become more energetic, they generally also become louder. This gives us another useful pair of features to consider when comparing songs.

In [ ]:
# One row per track-genre pair, dropping tracks with no genres
genre_df = data[~data["Missing Genres"]].explode("Genres").copy()
genre_df = genre_df.rename(columns={"Genres": "Genre"})

# Top 5 genres for each listener separately
top_genres = (
    genre_df.groupby("Listener")["Genre"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)
top_genres

In [ ]:
# Keep only each listener's own top 5 genres
top_pairs = set(top_genres.index)

genre_summary = (
    genre_df[
        [
            pair in top_pairs
            for pair in zip(genre_df["Listener"], genre_df["Genre"])
        ]
    ]
    .groupby(["Listener", "Genre"])[["Energy", "Valence"]]
    .mean()
    .reset_index()
)

genre_summary

In [ ]:
fig = px.bar(
    genre_summary,
    x="Genre",
    y=["Energy", "Valence"],
    barmode="group",
    facet_col="Listener",
    title="Average Energy and Valence by Top Genres"
)

fig.update_xaxes(matches=None)
fig.show()

The graph shows that the top genres have different average energy and valence levels. Hip hop has the highest average energy and valence, while R&B has the lowest average energy. Rap, melodic rap, and rage rap are closer together, but still show some differences. This suggests that audio features like energy and valence may help us compare genres and could be useful later when looking for possible subgenres.